In [1]:
from parkrun.data import load_clean_data

In [2]:
df = load_clean_data()

In [4]:
df.columns

Index(['Event Name', 'Event Date', 'Event Number', 'Position', 'Name',
       'Gender', 'Gender Position', 'Gender Total', 'Age Group', 'Age Grade %',
       'Time', 'Time Details', 'Achievement', 'Total parkruns',
       'Volunteer Count', 'Club Membership', 'Volunteer Club', 'Club', 'mins',
       'Runner_ID'],
      dtype='str')

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats as scipy_stats

## Linear Mixed Effects Model

$$\log(\text{mins}_{ijk}) = \beta_0 + \beta_{\text{course}_j} + \gamma_{\text{month}_k} + u_i + \varepsilon_{ijk}$$

| Term | Meaning | Data column |
|---|---|---|
| $\beta_{\text{course}_j}$ | Fixed effect for each parkrun course | `Event Name` |
| $\gamma_{\text{month}_k}$ | Fixed effect for seasonality | month extracted from `Event Date` |
| $u_i \sim \mathcal{N}(0, \sigma_u^2)$ | Random intercept for runner ability | `Runner_ID` |
| $\varepsilon_{ijk}$ | Residual error | — |

In [4]:
# --- Prepare modelling dataset ---
df_model = df.copy()

# Target variable: log(mins)
df_model["log_mins"] = np.log(df_model["mins"])

# Seasonality feature: zero-padded month string so sorting is chronological
df_model["month"] = (
    df_model["Event Date"].apply(lambda d: d.month).astype(str).str.zfill(2)
)

# Rename to avoid spaces in statsmodels formula strings
df_model = df_model.rename(columns={"Event Name": "course"})

# Only keep runners with ≥ 3 runs so the runner random effect is estimable
run_counts = df_model["Runner_ID"].value_counts()
eligible = run_counts[run_counts >= 3].index
df_model = df_model[df_model["Runner_ID"].isin(eligible)].copy()

print(f"Rows:             {len(df_model):>10,}")
print(f"Unique runners:   {df_model['Runner_ID'].nunique():>10,}")
print(f"Unique courses:   {df_model['course'].nunique():>10,}")
print(f"Months present:   {sorted(df_model['month'].unique())}")
print(
    f"\nMean log(mins):   {df_model['log_mins'].mean():.4f}  →  {np.exp(df_model['log_mins'].mean()):.1f} min"
)
print(f"Std  log(mins):   {df_model['log_mins'].std():.4f}")

Rows:              8,225,761
Unique runners:      671,577
Unique courses:          819
Months present:   ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

Mean log(mins):   3.3861  →  29.6 min
Std  log(mins):   0.2379


In [7]:
# --- Sample to a computationally feasible size ---
# We keep *all runs* of a random sample of runners (preserves within-runner
# repeated measures structure that identifies σ²_u).
N_RUNNERS = 8_000

rng = np.random.default_rng(42)
unique_runners = df_model["Runner_ID"].unique()

if len(unique_runners) > N_RUNNERS:
    sampled_ids = rng.choice(unique_runners, size=N_RUNNERS, replace=False)
    df_fit = df_model[df_model["Runner_ID"].isin(sampled_ids)].reset_index(drop=True)
else:
    df_fit = df_model.reset_index(drop=True)

# Groups must be a string/categorical for statsmodels
df_fit["runner_grp"] = df_fit["Runner_ID"].astype(str)

print(
    f"Fitting on {len(df_fit):,} rows  |  "
    f"{df_fit['runner_grp'].nunique():,} runners  |  "
    f"{df_fit['course'].nunique():,} courses  |  "
    f"{df_fit['month'].nunique()} months"
)

Fitting on 98,335 rows  |  8,000 runners  |  819 courses  |  12 months


In [ ]:
# --- Fit the Linear Mixed Effects Model ---
#
#   log(mins_ijk) = β0 + β_course_j + γ_month_k + u_i + ε_ijk
#
#   Fixed effects : C(course) + C(month)
#   Random effect : scalar intercept per runner  (groups = runner_grp)
#   Estimation    : REML via L-BFGS
#
# Note: reference levels are the alphabetically first course and month "01"

print("Fitting model — this may take a few minutes …")

lme = smf.mixedlm(
    "log_mins ~ C(course) + C(month)", data=df_fit, groups=df_fit["runner_grp"]
)
result = lme.fit(method="lbfgs", maxiter=100, disp=True)

print("Done.\n")
print(result.summary())

Fitting model — this may take a few minutes …


In [ ]:
# --- Variance Components ---
sigma2_u = float(result.cov_re.iloc[0, 0])  # runner random-effect variance
sigma2_e = float(result.scale)  # residual variance
icc = sigma2_u / (sigma2_u + sigma2_e)  # intraclass correlation coefficient

print(
    f"Runner variance   σ²_u = {sigma2_u:.6f}  →  SD = {np.sqrt(sigma2_u):.4f} log-min"
)
print(
    f"Residual variance σ²_ε = {sigma2_e:.6f}  →  SD = {np.sqrt(sigma2_e):.4f} log-min"
)
print(f"ICC (runner ability explains) = {icc*100:.1f}% of total variance")
print()

# Convert SD of runner random effect to a time multiplier
sd_u = np.sqrt(sigma2_u)
print("Runner-ability interpretation (on original time scale):")
print(
    f"  A runner 1 SD *faster* than average takes exp(-{sd_u:.4f}) ≈ {np.exp(-sd_u):.3f}× the reference time"
)
print(f"  i.e. roughly {(1 - np.exp(-sd_u))*100:.1f}% faster in minutes")
print(
    f"  A runner 1 SD *slower* than average takes exp(+{sd_u:.4f}) ≈ {np.exp(+sd_u):.3f}× the reference time"
)

# Pie chart of variance partition
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [sigma2_u, sigma2_e],
    labels=[
        f"Runner ability\n$\\sigma^2_u$ = {sigma2_u:.4f}\n({icc*100:.1f}%)",
        f"Residual\n$\\sigma^2_\\varepsilon$ = {sigma2_e:.4f}\n({(1-icc)*100:.1f}%)",
    ],
    colors=["steelblue", "coral"],
    startangle=90,
    autopct=None,
    wedgeprops=dict(edgecolor="white", linewidth=2),
)
ax.set_title("Variance Partition", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Course Fixed Effects (β_course) ---
fe = result.fe_params.copy()

# Extract all course coefficients
course_fe = fe[fe.index.str.startswith("C(course)[T.")]
course_fe.index = course_fe.index.str.removeprefix("C(course)[T.").str.removesuffix("]")

# Convert log-coefficient → % difference vs reference course
# (positive = slower, negative = faster)
course_pct = (np.exp(course_fe) - 1) * 100
course_pct_sorted = course_pct.sort_values()

# Add reference course at 0%
ref_course = (
    df_fit["course"].cat.categories[0]
    if hasattr(df_fit["course"], "cat")
    else sorted(df_fit["course"].unique())[0]
)
all_course_pct = pd.concat([pd.Series({ref_course: 0.0}), course_pct_sorted])
all_course_pct = all_course_pct.sort_values()

print(f"Total courses with estimated effect: {len(course_pct):,}")
print(f"Fastest course: {all_course_pct.index[0]}  ({all_course_pct.iloc[0]:+.1f}%)")
print(f"Slowest course: {all_course_pct.index[-1]}  ({all_course_pct.iloc[-1]:+.1f}%)")

N = 20
fig, axes = plt.subplots(1, 2, figsize=(17, 7))

for ax, data, title, color in [
    (axes[0], all_course_pct.head(N), f"{N} Fastest Courses", "steelblue"),
    (axes[1], all_course_pct.tail(N).iloc[::-1], f"{N} Slowest Courses", "firebrick"),
]:
    y_pos = range(len(data))
    ax.barh(y_pos, data.values, color=color, alpha=0.85, edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(data.index, fontsize=8)
    ax.set_xlabel("% difference vs reference course")
    ax.set_title(title, fontsize=11)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")

fig.suptitle(
    "Course Fixed Effects $\\beta_{\\mathrm{course}}$\n"
    "(controlled for runner ability and seasonality)",
    fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
# --- Seasonality Fixed Effects (γ_month) ---
month_fe = fe[fe.index.str.startswith("C(month)[T.")]
month_fe.index = month_fe.index.str.removeprefix("C(month)[T.").str.removesuffix("]")

# Convert to % difference, then add the reference month at 0
month_pct = (np.exp(month_fe) - 1) * 100
ref_month = sorted(df_fit["month"].unique())[0]
month_pct = pd.concat([pd.Series({ref_month: 0.0}), month_pct]).sort_index()

# Map "01"→"Jan" etc.
import calendar

month_labels = {str(m).zfill(2): calendar.month_abbr[m] for m in range(1, 13)}
xtick_labels = [month_labels.get(m, m) for m in month_pct.index]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["steelblue" if v <= 0 else "coral" for v in month_pct.values]
ax.bar(
    range(len(month_pct)), month_pct.values, color=colors, alpha=0.85, edgecolor="white"
)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(range(len(month_pct)))
ax.set_xticklabels(xtick_labels, rotation=0)
ax.set_ylabel("% slower (+) / faster (−) vs January")
ax.set_title(
    "Seasonality Fixed Effect $\\gamma_{\\mathrm{month}}$\n"
    "(blue = faster than Jan, red = slower than Jan)"
)
plt.tight_layout()
plt.show()

print("Month effects (% vs January):")
for m, v in zip(month_pct.index, month_pct.values):
    print(f"  {month_labels.get(m, m)}: {v:+.2f}%")

In [ ]:
# --- Runner Random Effects  u_i  (BLUPs) ---
# result.random_effects is a dict  {group_label: DataFrame(row per random effect)}
re_vals = pd.Series(
    {k: float(v.iloc[0]) for k, v in result.random_effects.items()}, name="u_i"
)

print(f"Runner random effects: N={len(re_vals):,}")
print(f"  Mean  : {re_vals.mean():.6f}  (should be ≈0)")
print(f"  Std   : {re_vals.std():.6f}  (should be ≈ √σ²_u = {np.sqrt(sigma2_u):.6f})")
print(f"  Skew  : {scipy_stats.skew(re_vals):.4f}")
print(f"  Kurt  : {scipy_stats.kurtosis(re_vals):.4f}  (excess, 0 = Normal)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram with theoretical Normal overlay
x_grid = np.linspace(re_vals.min(), re_vals.max(), 300)
pdf = scipy_stats.norm.pdf(x_grid, 0, np.sqrt(sigma2_u))
ax = axes[0]
ax.hist(
    re_vals,
    bins=100,
    density=True,
    color="steelblue",
    alpha=0.7,
    edgecolor="white",
    label="Empirical BLUPs",
)
ax.plot(
    x_grid,
    pdf,
    color="firebrick",
    linewidth=2,
    label=f"$\\mathcal{{N}}(0,\\,{sigma2_u:.4f})$",
)
ax.set_xlabel("Random intercept $u_i$ (log-mins)")
ax.set_ylabel("Density")
ax.set_title(f"Distribution of Runner Random Effects\n" f"N = {len(re_vals):,}")
ax.legend()

# Normal Q-Q plot
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(re_vals, dist="norm")
ax = axes[1]
ax.scatter(osm, osr, s=1, alpha=0.25, color="steelblue", rasterized=True)
ax.plot(osm, slope * np.array(osm) + intercept, color="firebrick", linewidth=1.5)
ax.set_xlabel("Theoretical quantiles")
ax.set_ylabel("Sample quantiles")
ax.set_title(f"Q-Q Plot  (r = {r:.4f})")

plt.suptitle("Runner Random Effects $u_i$", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Residual Diagnostics  ε_ijk ---
fitted = result.fittedvalues
resid = result.resid

print(f"Residuals — mean: {resid.mean():.6f},  std: {resid.std():.6f}")
print(f"  Skewness : {scipy_stats.skew(resid):.4f}")
print(f"  Kurtosis : {scipy_stats.kurtosis(resid):.4f}  (excess, 0 = Normal)")

# Subsample for scatter plot speed
plot_idx = resid.sample(n=min(50_000, len(resid)), random_state=42).index

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# 1 Residuals vs Fitted
ax = axes[0]
ax.scatter(
    fitted[plot_idx],
    resid[plot_idx],
    s=1,
    alpha=0.08,
    color="steelblue",
    rasterized=True,
)
ax.axhline(0, color="firebrick", linewidth=1.2)
ax.set_xlabel("Fitted $\\hat{y}$ (log-mins)")
ax.set_ylabel("Residual $\\varepsilon_{ijk}$")
ax.set_title("Residuals vs Fitted")

# 2 Histogram of residuals
ax = axes[1]
x_grid = np.linspace(resid.min(), resid.max(), 300)
ax.hist(resid, bins=120, density=True, color="steelblue", alpha=0.7, edgecolor="white")
ax.plot(
    x_grid,
    scipy_stats.norm.pdf(x_grid, 0, np.sqrt(sigma2_e)),
    color="firebrick",
    linewidth=2,
    label=f"$\\mathcal{{N}}(0,\\,{sigma2_e:.4f})$",
)
ax.set_xlabel("Residual $\\varepsilon_{ijk}$")
ax.set_ylabel("Density")
ax.set_title("Residual Distribution")
ax.legend()

# 3 Q-Q plot of residuals
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(
    resid.sample(n=min(50_000, len(resid)), random_state=42), dist="norm"
)
ax = axes[2]
ax.scatter(osm, osr, s=1, alpha=0.25, color="steelblue", rasterized=True)
ax.plot(osm, slope * np.array(osm) + intercept, color="firebrick", linewidth=1.5)
ax.set_xlabel("Theoretical quantiles")
ax.set_ylabel("Sample quantiles")
ax.set_title(f"Residual Q-Q Plot  (r = {r:.4f})")

plt.suptitle("Residual Diagnostics$\\quad\\varepsilon_{ijk}$", fontsize=12)
plt.tight_layout()
plt.show()